## Exp2: CGM + time_since_meal

In [11]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [12]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import CGMTimePredictor

MODEL_NAME = 'CGM+time_since_meal (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = None

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_cgm_time'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_stability_summary.json'

In [13]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 5)]


In [14]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, target = batch
    return model(cgm.to(device), tsm.to(device), has_meal.to(device)), target.to(device)

In [15]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)  # before constructing the model
    model = CGMTimePredictor(**{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_cgm_time' /f'gated_cgm_time_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — CGM+time_since_meal (gated) | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: CGMTimePredictor  params: 1,026,261


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.76 | MAE 7.33 | gRMSE 11.76 | Zone A 97.44% | A+B 99.80%
 30 min | RMSE 17.22 | MAE 11.58 | gRMSE 19.66 | Zone A 91.82% | A+B 99.62%
 60 min | RMSE 25.93 | MAE 17.23 | gRMSE 31.32 | Zone A 82.60% | A+B 99.13%
 90 min | RMSE 31.85 | MAE 21.27 | gRMSE 39.50 | Zone A 76.07% | A+B 98.47%
120 min | RMSE 35.30 | MAE 23.80 | gRMSE 44.36 | Zone A 72.22% | A+B 98.01%

--------------------------------------------------------------------------------
Test results — CGM+time_since_meal (gated) | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: CGMTimePredictor  params: 1,026,261


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.70 | MAE 7.32 | gRMSE 11.82 | Zone A 97.45% | A+B 99.82%
 30 min | RMSE 17.19 | MAE 11.61 | gRMSE 19.78 | Zone A 91.91% | A+B 99.61%
 60 min | RMSE 26.20 | MAE 17.41 | gRMSE 31.79 | Zone A 82.24% | A+B 99.19%
 90 min | RMSE 32.30 | MAE 21.61 | gRMSE 40.16 | Zone A 75.47% | A+B 98.39%
120 min | RMSE 35.83 | MAE 24.23 | gRMSE 45.07 | Zone A 71.14% | A+B 97.94%

--------------------------------------------------------------------------------
Test results — CGM+time_since_meal (gated) | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: CGMTimePredictor  params: 1,026,261


...

 15 min | RMSE 10.77 | MAE 7.43 | gRMSE 12.03 | Zone A 97.57% | A+B 99.78%
 30 min | RMSE 17.41 | MAE 11.71 | gRMSE 20.32 | Zone A 91.80% | A+B 99.55%
 60 min | RMSE 26.42 | MAE 17.37 | gRMSE 32.40 | Zone A 82.56% | A+B 99.01%
 90 min | RMSE 32.53 | MAE 21.52 | gRMSE 40.87 | Zone A 75.99% | A+B 98.36%
120 min | RMSE 36.15 | MAE 24.09 | gRMSE 45.97 | Zone A 71.84% | A+B 97.67%

--------------------------------------------------------------------------------
Test results — CGM+time_since_meal (gated) | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: CGMTimePredictor  params: 1,026,261


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.68 | MAE 7.31 | gRMSE 11.71 | Zone A 97.53% | A+B 99.79%
 30 min | RMSE 17.23 | MAE 11.57 | gRMSE 19.80 | Zone A 91.66% | A+B 99.58%
 60 min | RMSE 26.00 | MAE 17.26 | gRMSE 31.54 | Zone A 82.80% | A+B 99.11%
 90 min | RMSE 32.00 | MAE 21.32 | gRMSE 39.84 | Zone A 76.39% | A+B 98.47%
120 min | RMSE 35.54 | MAE 23.84 | gRMSE 44.86 | Zone A 72.19% | A+B 97.98%

--------------------------------------------------------------------------------
Test results — CGM+time_since_meal (gated) | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: CGMTimePredictor  params: 1,026,261


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.95 | MAE 7.46 | gRMSE 12.34 | Zone A 97.47% | A+B 99.79%
 30 min | RMSE 17.41 | MAE 11.72 | gRMSE 20.19 | Zone A 91.62% | A+B 99.57%
 60 min | RMSE 26.21 | MAE 17.42 | gRMSE 31.89 | Zone A 82.03% | A+B 99.08%
 90 min | RMSE 32.16 | MAE 21.43 | gRMSE 40.11 | Zone A 75.88% | A+B 98.38%
120 min | RMSE 35.59 | MAE 23.83 | gRMSE 44.97 | Zone A 71.89% | A+B 97.86%

--------------------------------------------------------------------------------


In [16]:
# Test Results
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [r['test_results'][h]['ceg']['A'] for r in all_results]
        elif m == 'zone_ab':
            seed_values = [r['test_results'][h]['ceg']['A'] + r['test_results'][h]['ceg']['B']
                            for r in all_results]
        else:
            seed_values = [r['test_results'][h][m] for r in all_results]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [_extract(r['test_results_per_participant'][pid][h], m)
                                    for r in all_results]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}

print(f'=== {MODEL_NAME} test set: overall mean +/- SD_seed (SD_participant) ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME, 'seeds': SEEDS, 'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS, 'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')

=== CGM+time_since_meal (gated) test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 10.76 +/- 0.10 (1.76) mg/dL   MAE = 7.38 +/- 0.08 (1.42)   gRMSE = 11.91 +/- 0.19 (2.86)   Zone A = 97.48 +/- 0.07 (1.88)%   Zone A+B = 99.79 +/- 0.01 (0.33)%
 30-min:  RMSE = 17.27 +/- 0.09 (3.27) mg/dL   MAE = 11.64 +/- 0.06 (2.41)   gRMSE = 19.90 +/- 0.21 (5.70)   Zone A = 91.75 +/- 0.17 (3.24)%   Zone A+B = 99.58 +/- 0.02 (0.63)%
 60-min:  RMSE = 26.17 +/- 0.17 (8.15) mg/dL   MAE = 17.36 +/- 0.07 (5.22)   gRMSE = 31.78 +/- 0.32 (13.19)   Zone A = 82.33 +/- 0.27 (3.24)%   Zone A+B = 99.12 +/- 0.06 (0.92)%
 90-min:  RMSE = 32.20 +/- 0.23 (12.46) mg/dL   MAE = 21.44 +/- 0.13 (7.74)   gRMSE = 40.12 +/- 0.39 (19.54)   Zone A = 75.82 +/- 0.31 (5.49)%   Zone A+B = 98.41 +/- 0.05 (1.59)%
120-min:  RMSE = 35.69 +/- 0.28 (15.39) mg/dL   MAE = 23.96 +/- 0.16 (9.75)   gRMSE = 45.01 +/- 0.48 (23.85)   Zone A = 71.85 +/- 0.39 (7.13)%   Zone A+B = 97.92 +/- 0.10 (2.65)%

saved per-seed results